In [ ]:
#r "nuget: Azure.AI.OpenAI"
#r "nuget: Azure.Identity"
#r "nuget: Azure"
#r "nuget: Newtonsoft.Json"
#r "nuget: Microsoft.Azure.Cosmos"

using Azure;
using Azure.AI.OpenAI;
using Azure.Identity;
using OpenAI.Embeddings;
using Newtonsoft.Json;
using Newtonsoft.Json.Linq;
using System.Net.Http;
using System.Collections.ObjectModel;
using Microsoft.Azure.Cosmos;

In [ ]:
public class Post    
{
    public string id { get; set; }
    public int PostId { get; set; }
    public string PostBody { get; set; }
    public string Title { get; set; }
    public int ViewCount { get; set; }
    public int AnswerCount { get; set; }
    public int CommentCount { get; set; }
    public int FavoriteCount { get; set; }
    public int AcceptedAnswerId { get; set; }
    public DateTime? CreatedOn { get; set; }
    public DateTime? ClosedDate { get; set; }
    public int OwnerUserId { get; set; }
    public string OwnerDisplayName { get; set; }
    public string PostType { get; set; }
    public int Score { get; set; }
    public string Tags { get; set; }
    public float[] postvector {get;set;}
    public string score {get;set;}
}

In [ ]:
var openAIClient = new AzureOpenAIClient(
    new Uri(""),
    new AzureKeyCredential(""));
var aiclient = openAIClient.GetEmbeddingClient("text-embedding-3-small");

In [ ]:
ReadOnlyMemory<float> GenerateVector(string text)
{    
    OpenAIEmbedding newembedding = aiclient.GenerateEmbedding(text);
    return newembedding.ToFloats();
}

In [ ]:
Console.WriteLine(string.Join(",",GenerateVector("This is a test embedding").ToArray()));

In [ ]:
var readcstring = "";
var readclient = new CosmosClient(readcstring);
var readdb = readclient.GetDatabase("Stackoverflow");
var readpostContainer = readdb.GetContainer("UserPosts");

In [ ]:
List<Post> ReadPostsLowerThan(int owneruserid)
{
    var productQuery = new QueryDefinition("SELECT top 500 * FROM c WHERE c.OwnerUserId = @id ORDER BY c.OwnerUserId")
        .WithParameter("@id", owneruserid);
    var iterator = readpostContainer.GetItemQueryIterator<Post>(productQuery);
    var results = new List<Post>();
    while (iterator.HasMoreResults)
    {
        var response = iterator.ReadNextAsync().Result;
        results.AddRange(response);    
    }
    return results;
}

In [ ]:
ReadPostsLowerThan(2).Count()

In [ ]:
var writecstring = "";
var writeclient = new CosmosClient(writecstring, new CosmosClientOptions() {AllowBulkExecution = true, EnableContentResponseOnWrite  = false});
var writedb = writeclient.GetDatabase("Stackoverflow");
var writepostContainer = writedb.GetContainer("PostsVectors");

In [ ]:
var id = 80;
for (int i = 200; i < 300; i++)
{
    var posts = ReadPostsLowerThan(i);
    foreach (var post in posts)
    {
        if (!string.IsNullOrEmpty(post.PostBody)){
        post.postvector = GenerateVector(post.PostBody).ToArray();
        var response = writepostContainer.UpsertItemAsync<Post>(post, new PartitionKey(post.OwnerUserId)).Result;
        Console.WriteLine($"Upserted post id: {post.OwnerUserId} with RU charge: {response.RequestCharge}");
        }
    }
}